# RAG-ассистент Фильмоскоп

Минимальный проектный notebook. Текущая реализация закрывает шаги 3–4 плана: два способа чанкинга, эмбеддинги, FAISS на CPU по умолчанию, baseline retrieval, retrieval-метрики, выбор лучшей стратегии и диагностику candidate pool. GPU FAISS используется только опционально, если он доступен в окружении.


## Константы и импорты


In [1]:
import json
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_PATH = ROOT / "filmoskop_kb.json"
EMBED_MODEL = "intfloat/multilingual-e5-base"
RECALL_KS = [1, 3, 5, 10, 20]
TOP_K = 20

## Проверка зависимостей для шага 3


In [2]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Не найдена зависимость: langchain-text-splitters. "
        "Установите её в активное окружение/Jupyter kernel: "
        "python -m pip install langchain-text-splitters"
    ) from e

from bs4 import BeautifulSoup
from ftfy import fix_text
import trafilatura
from sentence_transformers import SentenceTransformer
import faiss
import torch

print("Версия FAISS:", getattr(faiss, "__version__", "неизвестно"))

FAISS_GPU_AVAILABLE = (
    hasattr(faiss, "StandardGpuResources")
    and hasattr(faiss, "get_num_gpus")
    and faiss.get_num_gpus() > 0
)

if FAISS_GPU_AVAILABLE:
    print("Доступно GPU для FAISS:", faiss.get_num_gpus())
else:
    print("GPU FAISS недоступен; используется CPU FAISS.")

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство PyTorch для эмбеддингов:", TORCH_DEVICE)


Версия FAISS: 1.16.0
GPU FAISS недоступен; используется CPU FAISS.
Устройство PyTorch для эмбеддингов: cuda


## Диагностика датасета и базовая очистка для шага 3

На этом шаге загружаем корпус и запросы, считаем базовую статистику и проверяем качество сырого текста: остатки HTML, HTML entities, символы замены Unicode и mojibake. Для очистки используем `ftfy`, `trafilatura`/`BeautifulSoup` для HTML и регулярную очистку Markdown/text.


In [3]:
with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

docs = data["docs"]
queries = data["queries"]

print("Документы:", len(docs), "| Вопросы:", len(queries))
print("Форматы:", dict(Counter(d["format"] for d in docs)))
print("Категории:", dict(Counter(d["category"] for d in docs)))
print("Типы вопросов:", dict(Counter(q["kind"] for q in queries)))


Документы: 181 | Вопросы: 66
Форматы: {'text': 132, 'markdown': 28, 'html': 21}
Категории: {'Подписка и оплата': 38, 'Функции и контент': 34, 'Аккаунт и профили': 35, 'Устройства и качество': 42, 'Ошибки воспроизведения': 32}
Типы вопросов: {'exact': 15, 'semantic': 25, 'multi': 16, 'gap': 10}


In [4]:
HTML_NOISE_SELECTORS = [
    "script", "style", "noscript", "nav", "footer", "header",
    ".cookie-banner", ".breadcrumbs", ".main-menu",
]

QUALITY_PATTERNS = {
    "html_tags": re.compile(r"<[^>]+>"),
    "html_entities": re.compile(r"&(?:[a-zA-Z]+|#\d+|#x[0-9a-fA-F]+);"),
    "replacement_char": re.compile("\ufffd"),
    "latin_mojibake": re.compile(r"[\u00c3\u00c2][\x80-\xbf]|[\u00d0\u00d1][\x80-\xbf]"),
    "cyrillic_latin1_mojibake": re.compile(r"[\u00d0\u00d1][\u0080-\u00ff]"),
}


def count_quality_issues(items, text_key: str):
    """Считает документы с типовыми проблемами качества текста.

    Args:
        items: Список словарей с `doc_id` и текстовым полем.
        text_key: Имя поля, в котором находится проверяемый текст.

    Returns:
        `pandas.DataFrame` с числом документов по каждому типу проблемы и примерами `doc_id`.
    """
    rows = []
    for issue, pattern in QUALITY_PATTERNS.items():
        hit_ids = [item["doc_id"] for item in items if pattern.search(item.get(text_key, "") or "")]
        rows.append({"issue": issue, "docs": len(hit_ids), "doc_ids_sample": hit_ids[:20]})
    return pd.DataFrame(rows)


def html_to_text(raw: str) -> str:
    """Извлекает основной текст из HTML-документа.

    Args:
        raw: Сырой HTML или текст с HTML-разметкой.

    Returns:
        Текст без служебной разметки, навигации, cookie-баннеров, скриптов и стилей.
    """
    extracted = trafilatura.extract(
        raw,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    if extracted and len(extracted.strip()) >= 30:
        return extracted

    soup = BeautifulSoup(raw, "lxml")
    for selector in HTML_NOISE_SELECTORS:
        for node in soup.select(selector):
            node.decompose()
    main_node = soup.select_one("article.kb-article") or soup.select_one("main") or soup.body or soup
    return main_node.get_text(" ", strip=True)


def clean_markdown(text: str) -> str:
    """Удаляет базовую Markdown-разметку, сохраняя полезный текст.

    Args:
        text: Сырой markdown/plain text документ.

    Returns:
        Текст без ссылочной/структурной markdown-разметки.
    """
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"(?m)^\s{0,3}#{1,6}\s*", "", text)
    text = re.sub(r"(?m)^\s*[-*+]\s+", "", text)
    text = re.sub(r"[`*_>]+", " ", text)
    return text


def clean_text(raw: str, fmt: str = "") -> str:
    """Очищает документ и нормализует текст для retrieval.

    Args:
        raw: Исходный сырой документ из датасета.
        fmt: Формат документа из метаданных: `html`, `markdown` или `text`.

    Returns:
        Очищенный текст без HTML/Markdown-разметки, с исправлением mojibake через `ftfy`.
    """
    text = fix_text(raw or "")
    if fmt == "html" or re.search(r"<(?:html|body|div|p|article|main|h1|nav|br)\b", text, flags=re.IGNORECASE):
        text = html_to_text(text)
    text = clean_markdown(text)
    text = fix_text(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


raw_quality = count_quality_issues(docs, "raw")

clean_docs = []
for d in docs:
    clean_docs.append({
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "text": clean_text(d["raw"], d.get("format", "")),
    })

clean_quality = count_quality_issues(clean_docs, "text")
quality_comparison = raw_quality.merge(
    clean_quality,
    on="issue",
    suffixes=("_raw", "_clean"),
)

print("Сравнение качества: сырой текст против очищенного")
display(quality_comparison)

sample_issue_doc_ids = [47, 55, 120, 169, 173]
cleaning_examples = []
for doc_id in sample_issue_doc_ids:
    raw_doc = next((d for d in docs if d["doc_id"] == doc_id), None)
    clean_doc = next((d for d in clean_docs if d["doc_id"] == doc_id), None)
    if raw_doc is None or clean_doc is None:
        continue
    cleaning_examples.append({
        "doc_id": doc_id,
        "format": raw_doc["format"],
        "title": raw_doc["title"],
        "raw_sample": raw_doc["raw"][:260].replace("\n", " "),
        "clean_sample": clean_doc["text"][:360],
    })

pd.DataFrame(cleaning_examples)


Сравнение качества: сырой текст против очищенного


,issue,docs_raw,doc_ids_sample_raw,docs_clean,doc_ids_sample_clean
0,html_tags,21,"[8, 19, 23, 32, 47, 52, 57, 71, 74, 78, 87, 89...",0,[]
1,html_entities,7,"[47, 99, 110, 147, 154, 155, 173]",0,[]
2,replacement_char,1,[120],1,[120]
3,latin_mojibake,16,"[22, 38, 43, 46, 54, 55, 60, 73, 93, 106, 111,...",0,[]
4,cyrillic_latin1_mojibake,16,"[22, 38, 43, 46, 54, 55, 60, 73, 93, 106, 111,...",0,[]


,doc_id,format,title,raw_sample,clean_sample
0,47,html,Ошибка P-103: контент недоступен в регионе,<!DOCTYPE html><html><head><title>Ошибка P-103...,Ошибка P-103: контент недоступен в регионе Оши...
1,55,text,Восстановление пароля,"ÐÑÐ»Ð¸ Ð½Ðµ Ð¿Ð¾Ð»ÑÑÐ°ÐµÑÑÑ Ð²Ð¾Ð¹ÑÐ¸,...","Если не получается войти, нажмите «Забыли паро..."
2,120,text,Уведомления,«Фильмоскоп» присылает два типа уведомлений. С...,«Фильмоскоп» присылает два типа уведомлений. С...
3,169,text,Восстановление пароля (архив),Внимание: статья устарела и хранится в архиве....,Внимание: статья устарела и хранится в архиве....
4,173,html,Что делать при подозрении на взлом,<!DOCTYPE html><html><head><title>Что делать п...,Что делать при подозрении на взлом Что делать ...


## Шаг 3. Два варианта чанкинга


In [5]:
CHUNK_STRATEGIES = {
    "recursive_250_50": {"kind": "recursive", "chunk_size": 250, "chunk_overlap": 50},
    "recursive_350_70": {"kind": "recursive", "chunk_size": 350, "chunk_overlap": 70},
    "recursive_500_100": {"kind": "recursive", "chunk_size": 500, "chunk_overlap": 100},
    "recursive_900_150": {"kind": "recursive", "chunk_size": 900, "chunk_overlap": 150},
    "sentence_300_1": {"kind": "sentence", "max_chars": 300, "overlap_sentences": 1},
    "sentence_420_1": {"kind": "sentence", "max_chars": 420, "overlap_sentences": 1},
    "sentence_620_1": {"kind": "sentence", "max_chars": 620, "overlap_sentences": 1},
    "semantic_500_1": {"kind": "semantic", "max_chars": 500, "overlap_sentences": 1},
    "semantic_700_1": {"kind": "semantic", "max_chars": 700, "overlap_sentences": 1},
    "full_doc": {"kind": "full_doc"},
}


def split_sentences(text: str):
    """Делит русский текст на предложения простой regex-эвристикой.

    Args:
        text: Очищенный текст документа.

    Returns:
        Список непустых предложений или весь текст одним элементом.
    """
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?…])\s+(?=[А-ЯЁA-Z0-9«])", text)
    return [sentence.strip() for sentence in sentences if sentence.strip()] or ([text] if text else [])


def make_chunk(page_content: str, doc, chunk_id: int):
    """Создаёт chunk-словарь с контентом и retrieval-метаданными.

    Args:
        page_content: Текст чанка.
        doc: Очищенный документ.
        chunk_id: Порядковый номер чанка внутри документа.

    Returns:
        Словарь чанка для индексации и retrieval.
    """
    return {
        "page_content": page_content.strip(),
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "category": doc["category"],
        "chunk_id": chunk_id,
    }


def make_recursive_chunks(clean_docs, chunk_size: int, chunk_overlap: int):
    """Строит `RecursiveCharacterTextSplitter` чанки с заданными параметрами.

    Args:
        clean_docs: Очищенные документы.
        chunk_size: Максимальная длина чанка в символах.
        chunk_overlap: Перекрытие чанков в символах.

    Returns:
        Список чанков с метаданными.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "; ", ": ", " ", ""],
    )
    chunks = []
    for doc in clean_docs:
        source_text = f"{doc['title']}\n{doc['category']}\n{doc['text']}"
        for i, part in enumerate(splitter.split_text(source_text)):
            chunks.append(make_chunk(part, doc, i))
    return chunks


def make_sentence_window_chunks(clean_docs, max_chars: int, overlap_sentences: int = 1):
    """Строит короткие чанки по предложениям с sentence-overlap.

    Args:
        clean_docs: Очищенные документы.
        max_chars: Желаемая максимальная длина чанка.
        overlap_sentences: Число предложений из конца предыдущего чанка для overlap.

    Returns:
        Список sentence-based чанков.
    """
    chunks = []
    for doc in clean_docs:
        prefix = f"{doc['title']}. Категория: {doc['category']}. "
        sentences = split_sentences(doc["text"])
        start = 0
        chunk_id = 0
        while start < len(sentences):
            current = []
            size = len(prefix)
            end = start
            while end < len(sentences) and (not current or size + len(sentences[end]) + 1 <= max_chars):
                current.append(sentences[end])
                size += len(sentences[end]) + 1
                end += 1
            chunks.append(make_chunk(prefix + " ".join(current), doc, chunk_id))
            chunk_id += 1
            if end >= len(sentences):
                break
            start = max(start + 1, end - overlap_sentences)
    return chunks


def make_semantic_block_chunks(clean_docs, max_chars: int, overlap_sentences: int = 1):
    """Строит semantic-ish блоки по предложениям и маркерам смысловых границ.

    Args:
        clean_docs: Очищенные документы.
        max_chars: Желаемая максимальная длина смыслового блока.
        overlap_sentences: Число предложений overlap между соседними блоками.

    Returns:
        Список semantic-ish чанков: заголовок, категория и связный блок предложений.
    """
    boundary_re = re.compile(
        r"^(Важно|Шаг|Если |Для |Можно|Нельзя|Ошибка|Проблема|Решение|Ограничение|Подсказка)\b",
        flags=re.IGNORECASE,
    )
    chunks = []
    for doc in clean_docs:
        prefix = f"Заголовок: {doc['title']}\nКатегория: {doc['category']}\nТекст: "
        sentences = split_sentences(doc["text"])
        groups = []
        current = []
        size = len(prefix)
        for sentence in sentences:
            is_boundary = bool(boundary_re.search(sentence)) and current
            is_too_long = current and size + len(sentence) + 1 > max_chars
            if is_boundary or is_too_long:
                groups.append(current)
                current = current[-overlap_sentences:] if overlap_sentences else []
                size = len(prefix) + sum(len(item) + 1 for item in current)
            current.append(sentence)
            size += len(sentence) + 1
        if current:
            groups.append(current)
        for i, group in enumerate(groups):
            chunks.append(make_chunk(prefix + " ".join(group), doc, i))
    return chunks


def make_full_doc_chunks(clean_docs):
    """Создаёт один retrieval-чанк на документ.

    Args:
        clean_docs: Очищенные документы.

    Returns:
        Список document-level чанков.
    """
    return [
        make_chunk(
            f"Заголовок: {doc['title']}\nКатегория: {doc['category']}\nТекст: {doc['text']}",
            doc,
            0,
        )
        for doc in clean_docs
    ]


def make_chunks(clean_docs, strategy_config):
    """Диспетчеризует создание чанков по конфигурации стратегии.

    Args:
        clean_docs: Очищенные документы.
        strategy_config: Параметры стратегии из `CHUNK_STRATEGIES`.

    Returns:
        Список чанков выбранной стратегии.
    """
    kind = strategy_config["kind"]
    if kind == "recursive":
        return make_recursive_chunks(
            clean_docs,
            chunk_size=strategy_config["chunk_size"],
            chunk_overlap=strategy_config["chunk_overlap"],
        )
    if kind == "sentence":
        return make_sentence_window_chunks(
            clean_docs,
            max_chars=strategy_config["max_chars"],
            overlap_sentences=strategy_config["overlap_sentences"],
        )
    if kind == "semantic":
        return make_semantic_block_chunks(
            clean_docs,
            max_chars=strategy_config["max_chars"],
            overlap_sentences=strategy_config["overlap_sentences"],
        )
    if kind == "full_doc":
        return make_full_doc_chunks(clean_docs)
    raise ValueError(f"Неизвестный тип стратегии: {kind}")


chunk_sets = {}
chunk_stats = []
for name, params in CHUNK_STRATEGIES.items():
    chunk_sets[name] = make_chunks(clean_docs, params)
    lengths = [len(c["page_content"]) for c in chunk_sets[name]]
    chunk_stats.append({
        "strategy": name,
        "chunks": len(lengths),
        "median_len": int(np.median(lengths)),
        "p90_len": int(np.quantile(lengths, 0.90)),
    })

pd.DataFrame(chunk_stats)


,strategy,chunks,median_len,p90_len
0,recursive_250_50,911,173,233
1,recursive_350_70,687,248,327
2,recursive_500_100,469,377,484
3,recursive_900_150,309,472,871
4,sentence_300_1,1011,249,289
5,sentence_420_1,602,357,406
6,sentence_620_1,344,522,599
7,semantic_500_1,531,404,481
8,semantic_700_1,371,500,672
9,full_doc,181,652,1297


## Шаг 3. Эмбеддинги и FAISS-индексы


In [6]:
embedder = SentenceTransformer(EMBED_MODEL, device=TORCH_DEVICE)

def encode_passages(texts):
    """Создаёт векторы документов для модели E5.

    Args:
        texts: Список текстов чанков.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True,
    )
    return np.asarray(vecs, dtype="float32")

def encode_queries(texts):
    """Создаёт нормализованные векторы запросов для модели E5.

    Args:
        texts: Список текстов запросов.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["query: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    return np.asarray(vecs, dtype="float32")

def build_faiss_index(vectors):
    """Создаёт индекс FAISS `IndexFlatIP` для нормализованных эмбеддингов.

    Args:
        vectors: Матрица dense-эмбеддингов `float32` формы
            `(количество_чанков, размерность_эмбеддинга)`.

    Returns:
        Индекс FAISS с добавленными векторами. По умолчанию используется CPU
        FAISS; GPU FAISS используется только если он доступен в окружении.
    """
    cpu_index = faiss.IndexFlatIP(vectors.shape[1])
    if FAISS_GPU_AVAILABLE:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        index.add(vectors)
        return index

    cpu_index.add(vectors)
    return cpu_index

indexes = {}
embeddings = {}
for name, chunks in chunk_sets.items():
    embeddings[name] = encode_passages([c["page_content"] for c in chunks])
    indexes[name] = build_faiss_index(embeddings[name])
    print(name, embeddings[name].shape, "размер индекса", indexes[name].ntotal)


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

recursive_250_50 (911, 768) размер индекса 911


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

recursive_350_70 (687, 768) размер индекса 687


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

recursive_500_100 (469, 768) размер индекса 469


... (10 outputs left out)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

semantic_700_1 (371, 768) размер индекса 371


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

full_doc (181, 768) размер индекса 181


## Шаг 3. Retrieval-метрики


In [7]:
eval_queries = [q for q in queries if q["kind"] != "gap"]
query_vecs = encode_queries([q["query"] for q in eval_queries])

def retrieved_doc_ids(strategy: str, q_vec, k: int):
    """Возвращает уникальные `doc_id` из top-k найденных чанков.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
        q_vec: Вектор запроса.
        k: Количество чанков, запрашиваемое у FAISS.

    Returns:
        Список уникальных идентификаторов документов в порядке релевантности.
    """
    _, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    doc_ids = []
    seen = set()
    for i in idx[0]:
        doc_id = chunk_sets[strategy][int(i)]["doc_id"]
        if doc_id not in seen:
            doc_ids.append(doc_id)
            seen.add(doc_id)
    return doc_ids

def precision_at_k(found, relevant):
    """Считает precision для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля найденных документов, которые входят в список релевантных.
    """
    if not found:
        return 0.0
    rel = set(relevant)
    return sum(doc_id in rel for doc_id in found) / len(found)

def recall_at_k(found, relevant):
    """Считает recall для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля релевантных документов, найденных поиском.
    """
    rel = set(relevant)
    if not rel:
        return 0.0
    return len(set(found) & rel) / len(rel)

def mrr(found, relevant):
    """Считает reciprocal rank первого релевантного документа.

    Args:
        found: Список найденных идентификаторов документов в порядке релевантности.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Значение reciprocal rank или `0.0`, если релевантных документов нет в выдаче.
    """
    rel = set(relevant)
    for rank, doc_id in enumerate(found, start=1):
        if doc_id in rel:
            return 1 / rank
    return 0.0

def evaluate_strategy(strategy: str):
    """Считает retrieval-метрики для одной стратегии чанкинга.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.

    Returns:
        `pandas.DataFrame` по вопросам со списками найденных и релевантных документов.
    """
    rows = []
    for q, q_vec in zip(eval_queries, query_vecs):
        found20 = retrieved_doc_ids(strategy, q_vec, TOP_K)
        row = {
            "strategy": strategy,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found20,
            "precision@5": precision_at_k(found20[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found20[:5], q["relevant_docs"]),
            "MRR": mrr(found20, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found20[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

metrics_by_query = pd.concat([evaluate_strategy(name) for name in chunk_sets], ignore_index=True)
summary = metrics_by_query.groupby("strategy")[["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]].mean().reset_index()
summary


,strategy,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,full_doc,0.157143,0.616071,0.475850,0.232143,0.526786,0.857143,0.964286
1,recursive_250_50,0.157143,0.598214,0.566751,0.348214,0.580357,0.776786,0.857143
2,recursive_350_70,0.164286,0.642857,0.473073,0.232143,0.508929,0.857143,0.910714
3,recursive_500_100,0.167857,0.642857,0.487053,0.258929,0.562500,0.723214,0.803571
4,recursive_900_150,0.146429,0.562500,0.444048,0.205357,0.464286,0.687500,0.830357
5,semantic_500_1,0.196429,0.767857,0.636565,0.401786,0.696429,0.794643,0.830357
6,semantic_700_1,0.192857,0.741071,0.590744,0.366071,0.651786,0.830357,0.910714
7,sentence_300_1,0.182440,0.696429,0.588010,0.375000,0.589286,0.794643,0.812500
8,sentence_420_1,0.180357,0.696429,0.614606,0.401786,0.625000,0.821429,0.875000
9,sentence_620_1,0.175000,0.660714,0.559357,0.348214,0.589286,0.857143,0.928571


In [8]:
by_kind = metrics_by_query.groupby(["strategy", "kind"])[["precision@5", "recall@5", "MRR"]].mean().reset_index()
by_kind

,strategy,kind,precision@5,recall@5,MRR
0,full_doc,exact,0.186667,0.900000,0.583333
1,full_doc,multi,0.212500,0.531250,0.444444
2,full_doc,semantic,0.104000,0.500000,0.431459
3,recursive_250_50,exact,0.200000,0.966667,0.866667
4,recursive_250_50,multi,0.250000,0.625000,0.680155
5,recursive_250_50,semantic,0.072000,0.360000,0.314224
6,recursive_350_70,exact,0.200000,0.966667,0.711111
7,recursive_350_70,multi,0.225000,0.562500,0.497991
8,recursive_350_70,semantic,0.104000,0.500000,0.314303
9,recursive_500_100,exact,0.200000,0.966667,0.738889


## Шаг 3. Анализ примеров чанков и выбор стратегии


In [9]:
for name, chunks in chunk_sets.items():
    print("\n===", name, "===")
    for c in chunks[:3]:
        print(f"документ={c['doc_id']} чанк={c['chunk_id']} заголовок={c['title']}")
        print(c["page_content"][:450].replace("\n", " "))
        print("---")


=== recursive_250_50 ===
документ=0 чанк=0 заголовок=Промокоды
Промокоды Подписка и оплата
---
документ=0 чанк=1 заголовок=Промокоды
Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки
---
документ=0 чанк=2 заголовок=Промокоды
. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции
---

=== recursive_350_70 ===
документ=0 чанк=0 заголовок=Промокоды
Промокоды Подписка и оплата
---
документ=0 чанк=1 заголовок=Промокоды
Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции
---
документ=0 чанк=2

In [10]:
best_strategy = summary.sort_values(["recall@5", "MRR", "precision@5"], ascending=False).iloc[0]["strategy"]
print("Выбранная стратегия чанкинга:", best_strategy)
print(summary.to_string(index=False))


Выбранная стратегия чанкинга: semantic_500_1
         strategy  precision@5  recall@5      MRR  recall@1  recall@3  recall@10  recall@20
         full_doc     0.157143  0.616071 0.475850  0.232143  0.526786   0.857143   0.964286
 recursive_250_50     0.157143  0.598214 0.566751  0.348214  0.580357   0.776786   0.857143
 recursive_350_70     0.164286  0.642857 0.473073  0.232143  0.508929   0.857143   0.910714
recursive_500_100     0.167857  0.642857 0.487053  0.258929  0.562500   0.723214   0.803571
recursive_900_150     0.146429  0.562500 0.444048  0.205357  0.464286   0.687500   0.830357
   semantic_500_1     0.196429  0.767857 0.636565  0.401786  0.696429   0.794643   0.830357
   semantic_700_1     0.192857  0.741071 0.590744  0.366071  0.651786   0.830357   0.910714
   sentence_300_1     0.182440  0.696429 0.588010  0.375000  0.589286   0.794643   0.812500
   sentence_420_1     0.180357  0.696429 0.614606  0.401786  0.625000   0.821429   0.875000
   sentence_620_1     0.175000  0.6

### Вывод по шагу 3

После локального запуска нужно сравнить агрегированные метрики `summary` и `by_kind`, выбрать стратегию по `recall@5`, `MRR`, `precision@5` и зафиксировать причины выбора. Выбранная стратегия используется в шаге 4 для baseline retrieval и диагностики ошибок. Финальный контрольный прогон выполняется на удалённой машине только после успешного локального запуска.


## Шаг 4. Baseline retrieval и диагностика

На выбранной стратегии чанкинга собираем итоговый dense baseline: выдачу top-K чанков, распределение score для подбора порога, doc-level overfetch с агрегацией чанков в уникальные `doc_id`, пример фильтрации по категории и диагностику ошибок по типам запросов.

In [11]:
BASELINE_CHUNK_TOP_K = 20
OVERFETCH_CHUNK_KS = [50, 100]
DOC_TOP_K = 20
FILTER_CATEGORY = "Подписка и оплата"

selected_strategy = best_strategy
selected_chunks = chunk_sets[selected_strategy]
selected_index = indexes[selected_strategy]
doc_by_id = {doc["doc_id"]: doc for doc in clean_docs}

eval_query_rows = list(zip(eval_queries, query_vecs))

print("Стратегия для базового поиска:", selected_strategy)
print("Чанков в выбранной стратегии:", len(selected_chunks))
print("Оценочных запросов без gap:", len(eval_query_rows))

Стратегия для базового поиска: semantic_500_1
Чанков в выбранной стратегии: 531
Оценочных запросов без gap: 56


In [12]:
def dense_chunk_search(strategy: str, q_vec, chunk_k: int, category: str | None = None, score_threshold: float | None = None):
    """Ищет чанки через FAISS и возвращает результаты с метаданными.
    
        Args:
            strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
            q_vec: Вектор запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            category: Необязательная категория для постфильтрации результатов.
            score_threshold: Необязательный минимальный dense-score.
    
        Returns:
            Список словарей с rank, score, doc_id, title, category, chunk_id и page_content.
    """
    chunks = chunk_sets[strategy]
    k = min(chunk_k, len(chunks))
    scores, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    hits = []
    rank = 0
    for score, chunk_idx in zip(scores[0], idx[0]):
        if chunk_idx < 0:
            continue
        chunk = chunks[int(chunk_idx)]
        score = float(score)
        if category is not None and chunk["category"] != category:
            continue
        if score_threshold is not None and score < score_threshold:
            continue
        rank += 1
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": score,
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def aggregate_chunks_to_docs(chunk_hits, doc_k: int | None = None):
    """Агрегирует результаты уровня чанков в уникальные документы по лучшему score чанка.
    
        Args:
            chunk_hits: Результаты, вернутые `dense_chunk_search`.
            doc_k: Необязательное максимальное число документов в выдаче.
    
        Returns:
            Ранжированный список уникальных document hits.
    """
    docs_map = {}
    for hit in chunk_hits:
        doc_id = hit["doc_id"]
        if doc_id not in docs_map:
            docs_map[doc_id] = {
                "doc_id": doc_id,
                "title": hit["title"],
                "category": hit["category"],
                "best_score": hit["score"],
                "best_chunk_rank": hit["rank"],
                "best_chunk_id": hit["chunk_id"],
                "chunk_count": 1,
                "page_content": hit["page_content"],
            }
        else:
            docs_map[doc_id]["chunk_count"] += 1
            if hit["score"] > docs_map[doc_id]["best_score"]:
                docs_map[doc_id].update({
                    "best_score": hit["score"],
                    "best_chunk_rank": hit["rank"],
                    "best_chunk_id": hit["chunk_id"],
                    "page_content": hit["page_content"],
                })
    docs_ranked = sorted(
        docs_map.values(),
        key=lambda row: (-row["best_score"], row["best_chunk_rank"]),
    )
    if doc_k is not None:
        docs_ranked = docs_ranked[:doc_k]
    for rank, row in enumerate(docs_ranked, start=1):
        row["doc_rank"] = rank
    return docs_ranked


def dense_doc_ids_with_overfetch(strategy: str, q_vec, chunk_k: int, doc_k: int):
    """Возвращает top document id после overfetch chunk-кандидатов.
    
        Args:
            strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
            q_vec: Вектор запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированный список уникальных идентификаторов документов.
    """
    chunk_hits = dense_chunk_search(strategy, q_vec, chunk_k)
    doc_hits = aggregate_chunks_to_docs(chunk_hits, doc_k=doc_k)
    return [row["doc_id"] for row in doc_hits]


def evaluate_dense_variant(strategy: str, chunk_k: int, doc_k: int, variant: str):
    """Считает retrieval-метрики для dense-варианта с выбранным overfetch.
    
        Args:
            variant: Человекочитаемое название варианта.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            `pandas.DataFrame` с метриками варианта по каждому запросу.
    """
    rows = []
    for q, q_vec in eval_query_rows:
        found_docs = dense_doc_ids_with_overfetch(strategy, q_vec, chunk_k=chunk_k, doc_k=doc_k)
        row = {
            "variant": variant,
            "strategy": strategy,
            "chunk_k": chunk_k,
            "doc_k": doc_k,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found_docs,
            "precision@5": precision_at_k(found_docs[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found_docs[:5], q["relevant_docs"]),
            "MRR": mrr(found_docs, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found_docs[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

In [13]:
score_rows = []
for query_idx, (q, q_vec) in enumerate(eval_query_rows):
    for hit in dense_chunk_search(selected_strategy, q_vec, chunk_k=BASELINE_CHUNK_TOP_K):
        score_rows.append({
            "query_idx": query_idx,
            "kind": q["kind"],
            "chunk_rank": hit["rank"],
            "score": hit["score"],
            "doc_id": hit["doc_id"],
            "is_relevant_doc": hit["doc_id"] in set(q["relevant_docs"]),
        })

score_distribution = pd.DataFrame(score_rows)
score_quantiles = score_distribution["score"].quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).reset_index()
score_quantiles.columns = ["quantile", "score"]
DENSE_SCORE_THRESHOLD = float(score_distribution["score"].quantile(0.10))

print("Диагностический порог dense-score p10:", round(DENSE_SCORE_THRESHOLD, 4))
score_quantiles

Диагностический порог dense-score p10: 0.8066


,quantile,score
0,0.01,0.780244
1,0.05,0.800078
2,0.10,0.806572
3,0.25,0.819017
4,0.50,0.831558
5,0.75,0.846397
6,0.90,0.860199
7,0.95,0.866355
8,0.99,0.890178


In [14]:
sample_query, sample_vec = eval_query_rows[0]
sample_chunk_hits = dense_chunk_search(selected_strategy, sample_vec, chunk_k=10)
sample_doc_hits = aggregate_chunks_to_docs(sample_chunk_hits, doc_k=5)

print("Пример запроса:", sample_query["query"])
print("Эталонные doc_id:", sample_query["relevant_docs"])
pd.DataFrame(sample_chunk_hits)[["rank", "score", "doc_id", "title", "category", "chunk_id"]].head(10)

Пример запроса: что означает ошибка P-102
Эталонные doc_id: [85]


,rank,score,doc_id,title,category,chunk_id
0,1,0.895402,70,Ошибка P-102 (архив),Ошибки воспроизведения,0
1,2,0.876881,70,Ошибка P-102 (архив),Ошибки воспроизведения,2
2,3,0.874755,70,Ошибка P-102 (архив),Ошибки воспроизведения,1
3,4,0.872476,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,0
4,5,0.869149,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,1
5,6,0.867296,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,2
6,7,0.866126,178,Ошибка P-109: недостаточно места,Ошибки воспроизведения,0
7,8,0.865430,129,Ошибка P-105: устаревшее приложение,Ошибки воспроизведения,0
8,9,0.865029,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,3
9,10,0.864898,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,0


In [15]:
print("Топ-5 документов после агрегации попаданий чанков:")
pd.DataFrame(sample_doc_hits)[["doc_rank", "best_score", "doc_id", "title", "category", "best_chunk_id", "chunk_count"]]

Топ-5 документов после агрегации попаданий чанков:


,doc_rank,best_score,doc_id,title,category,best_chunk_id,chunk_count
0,1,0.895402,70,Ошибка P-102 (архив),Ошибки воспроизведения,0,3
1,2,0.872476,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,0,4
2,3,0.866126,178,Ошибка P-109: недостаточно места,Ошибки воспроизведения,0,1
3,4,0.865430,129,Ошибка P-105: устаревшее приложение,Ошибки воспроизведения,0,1
4,5,0.864898,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,0,1


In [16]:
baseline_variants = []
baseline_variants.append(evaluate_dense_variant(
    selected_strategy,
    chunk_k=BASELINE_CHUNK_TOP_K,
    doc_k=DOC_TOP_K,
    variant="dense_chunk20_doc20",
))
for chunk_k in OVERFETCH_CHUNK_KS:
    baseline_variants.append(evaluate_dense_variant(
        selected_strategy,
        chunk_k=chunk_k,
        doc_k=DOC_TOP_K,
        variant=f"dense_chunk{chunk_k}_doc20",
    ))

baseline_diagnostics = pd.concat(baseline_variants, ignore_index=True)
baseline_summary = baseline_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@20", "recall@5", "MRR"], ascending=False)
baseline_summary

,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_chunk100_doc20,0.196429,0.767857,0.641455,0.401786,0.696429,0.812500,0.964286
2,dense_chunk50_doc20,0.196429,0.767857,0.641455,0.401786,0.696429,0.812500,0.964286
1,dense_chunk20_doc20,0.196429,0.767857,0.636565,0.401786,0.696429,0.794643,0.830357


In [17]:
baseline_by_kind = baseline_diagnostics.groupby(["variant", "kind"])[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index()
baseline_by_kind.sort_values(["variant", "kind"])

,variant,kind,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_chunk100_doc20,exact,0.213333,1.00000,0.811111,0.666667,0.966667,1.0000,1.0000
1,dense_chunk100_doc20,multi,0.262500,0.65625,0.709673,0.281250,0.562500,0.7500,0.9375
2,dense_chunk100_doc20,semantic,0.144000,0.70000,0.496001,0.320000,0.620000,0.7400,0.9600
3,dense_chunk20_doc20,exact,0.213333,1.00000,0.811111,0.666667,0.966667,1.0000,1.0000
4,dense_chunk20_doc20,multi,0.262500,0.65625,0.709673,0.281250,0.562500,0.6875,0.7500
5,dense_chunk20_doc20,semantic,0.144000,0.70000,0.485048,0.320000,0.620000,0.7400,0.7800
6,dense_chunk50_doc20,exact,0.213333,1.00000,0.811111,0.666667,0.966667,1.0000,1.0000
7,dense_chunk50_doc20,multi,0.262500,0.65625,0.709673,0.281250,0.562500,0.7500,0.9375
8,dense_chunk50_doc20,semantic,0.144000,0.70000,0.496001,0.320000,0.620000,0.7400,0.9600


In [18]:
category_query_idx = None
for i, (q, q_vec) in enumerate(eval_query_rows):
    relevant_categories = {doc_by_id[doc_id]["category"] for doc_id in q["relevant_docs"] if doc_id in doc_by_id}
    if FILTER_CATEGORY in relevant_categories:
        category_query_idx = i
        break

if category_query_idx is None:
    category_query_idx = 0

category_query, category_vec = eval_query_rows[category_query_idx]
category_hits = dense_chunk_search(
    selected_strategy,
    category_vec,
    chunk_k=100,
    category=FILTER_CATEGORY,
    score_threshold=None,
)
category_doc_hits = aggregate_chunks_to_docs(category_hits, doc_k=5)

print("Категория фильтра:", FILTER_CATEGORY)
print("Запрос:", category_query["query"])
print("Эталонные doc_id:", category_query["relevant_docs"])
pd.DataFrame(category_doc_hits)[["doc_rank", "best_score", "doc_id", "title", "category", "best_chunk_id", "chunk_count"]]

Категория фильтра: Подписка и оплата
Запрос: сколько стоит тариф Максимум и что в него входит
Эталонные doc_id: [113]


,doc_rank,best_score,doc_id,title,category,best_chunk_id,chunk_count
0,1,0.865442,88,Смена тарифа (архив),Подписка и оплата,0,2
1,2,0.863304,24,Какой тариф подойдёт именно вам,Подписка и оплата,0,4
2,3,0.856782,113,Тарифы «Фильмоскопа»,Подписка и оплата,3,7
3,4,0.838164,25,Тарифы «Фильмоскопа» (архив),Подписка и оплата,0,2
4,5,0.833037,32,Почему зрители выбирают «Максимум»,Подписка и оплата,0,1


In [19]:
diagnostic_variant = "dense_chunk100_doc20" if "dense_chunk100_doc20" in set(baseline_diagnostics["variant"]) else baseline_summary.iloc[0]["variant"]
problem_rows = []

for q, q_vec in eval_query_rows:
    chunk_hits_100 = dense_chunk_search(selected_strategy, q_vec, chunk_k=100)
    doc_hits_100 = aggregate_chunks_to_docs(chunk_hits_100, doc_k=100)
    doc_positions = {row["doc_id"]: row["doc_rank"] for row in doc_hits_100}
    found20 = [row["doc_id"] for row in doc_hits_100[:DOC_TOP_K]]
    relevant = q["relevant_docs"]
    relevant_positions = {doc_id: doc_positions.get(doc_id) for doc_id in relevant}
    missing_relevant = [doc_id for doc_id, pos in relevant_positions.items() if pos is None]
    below_top5 = [doc_id for doc_id, pos in relevant_positions.items() if pos is not None and pos > 5]
    recall5 = recall_at_k(found20[:5], relevant)
    recall20 = recall_at_k(found20[:20], relevant)

    reasons = []
    if missing_relevant:
        reasons.append("relevant docs отсутствуют в top-100 candidate pool")
    if below_top5:
        reasons.append("relevant docs есть ниже top-5")
    if q["kind"] == "semantic":
        reasons.append("возможна сильная переформулировка запроса")
    if q["kind"] == "exact":
        reasons.append("dense-поиск может плохо ловить точные термины/коды")
    if q["kind"] == "multi":
        reasons.append("multi-запрос может требовать декомпозиции")

    if recall5 < 1.0:
        problem_rows.append({
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": relevant,
            "found_top5": found20[:5],
            "found_top20": found20[:20],
            "recall@5": recall5,
            "recall@20": recall20,
            "relevant_positions_in_top100_docs": relevant_positions,
            "diagnosis": "; ".join(reasons),
        })

problem_queries = pd.DataFrame(problem_rows).sort_values(["recall@5", "recall@20", "kind"]).reset_index(drop=True)
print("Проблемных запросов с recall@5 < 1.0:", len(problem_queries))
problem_queries.head(15)

Проблемных запросов с recall@5 < 1.0: 18


,kind,query,relevant_docs,found_top5,found_top20,recall@5,recall@20,relevant_positions_in_top100_docs,diagnosis
0,semantic,"в приложении всё включено, а на телефон ничего...",[124],"[72, 66, 172, 117, 21]","[72, 66, 172, 117, 21, 86, 129, 30, 41, 166, 1...",0.0,0.0,{124: 31},relevant docs есть ниже top-5; возможна сильна...
1,multi,заплатил за новинку отдельно — сколько она у м...,"[73, 140]","[116, 71, 40, 141, 103]","[116, 71, 40, 141, 103, 0, 3, 95, 84, 52, 29, ...",0.0,0.5,"{73: 14, 140: 27}",relevant docs есть ниже top-5; multi-запрос мо...
2,semantic,"как сделать, чтобы деньги сами не снимались ка...","[61, 64]","[180, 164, 114, 5, 155]","[180, 164, 114, 5, 155, 162, 64, 113, 116, 122...",0.0,1.0,"{61: 19, 64: 7}",relevant docs есть ниже top-5; возможна сильна...
3,semantic,"не пускает в аккаунт, пароль вылетел из головы",[55],"[9, 72, 71, 80, 117]","[9, 72, 71, 80, 117, 166, 173, 97, 21, 7, 130,...",0.0,1.0,{55: 17},relevant docs есть ниже top-5; возможна сильна...
4,semantic,"боюсь, что ребёнок случайно купит фильм с моей...",[140],"[108, 84, 73, 20, 71]","[108, 84, 73, 20, 71, 116, 96, 173, 171, 47, 1...",0.0,1.0,{140: 13},relevant docs есть ниже top-5; возможна сильна...
5,semantic,"как посмотреть, за что с меня списали деньги в...",[106],"[5, 64, 143, 162, 116]","[5, 64, 143, 162, 116, 173, 154, 115, 114, 71,...",0.0,1.0,{106: 15},relevant docs есть ниже top-5; возможна сильна...
6,semantic,"хочу убрать из ленты фильмы, которые уже смотрел",[97],"[20, 46, 81, 119, 157]","[20, 46, 81, 119, 157, 168, 71, 112, 108, 70, ...",0.0,1.0,{97: 12},relevant docs есть ниже top-5; возможна сильна...
7,semantic,"видят ли другие люди, что я поставил фильму лайк",[135],"[99, 157, 173, 41, 96]","[99, 157, 173, 41, 96, 24, 108, 134, 27, 81, 9...",0.0,1.0,{135: 14},relevant docs есть ниже top-5; возможна сильна...
8,multi,чем Стандарт отличается от Максимума и потеряю...,"[113, 128]","[88, 155, 128, 64, 28]","[88, 155, 128, 64, 28, 67, 162, 24, 71, 180, 9...",0.5,0.5,"{113: 21, 128: 3}",relevant docs есть ниже top-5; multi-запрос мо...
9,multi,сколько устройств могут смотреть одновременно ...,"[85, 113]","[44, 85, 70, 166, 24]","[44, 85, 70, 166, 24, 27, 17, 110, 99, 144, 13...",0.5,1.0,"{85: 2, 113: 20}",relevant docs есть ниже top-5; multi-запрос мо...


### Вывод по шагу 4

Шаг 4 добавляет итоговый dense baseline для выбранной стратегии чанкинга, диагностический порог по распределению similarity score, выдачу top-K чанков с метаданными, сравнение обычного chunk-level top-20 с doc-level overfetch top-50/top-100, пример фильтрации по категории `Подписка и оплата`, метрики по типам запросов и таблицу проблемных запросов. Если `recall@20` после overfetch остаётся ниже целевого уровня, следующий шаг — улучшать candidate generation через BM25, RRF и query transformations, а reranker подключать только после попадания релевантных документов в candidate pool.

## Шаг 5. Улучшение retrieval и выбор итоговой схемы поиска

На этом шаге улучшаем candidate generation относительно dense-only baseline: добавляем структурированный `searchable_text`, BM25, dense+BM25 RRF и простые rule-based трансформации запросов. После диагностики высокого `recall@20` и недостаточного `recall@5` дополнительно проверяем cross-encoder reranker для перестановки top-100 dense chunk-кандидатов.


In [20]:
from rank_bm25 import BM25Okapi

STEP5_CHUNK_K = 100
STEP5_DOC_K = 20
RRF_K = 60

def build_searchable_text(title: str, category: str, text: str) -> str:
    """Создаёт структурированный текст только для retrieval-индексации.
    
        Args:
            title: Заголовок документа.
            category: Категория документа.
            text: Исходный текст чанка.
    
        Returns:
            Структурированный searchable text с добавленными заголовком и категорией.
    """
    parts = [
        f"\u0417\u0430\u0433\u043e\u043b\u043e\u0432\u043e\u043a: {title}",
        f"\u041a\u0430\u0442\u0435\u0433\u043e\u0440\u0438\u044f: {category}",
        f"\u0422\u0435\u043a\u0441\u0442: {text}",
    ]
    return "\n".join(parts)


def add_searchable_text(chunks):
    """Добавляет `searchable_text`, сохраняя исходный `page_content`.
    
        Args:
            chunks: Исходные словари чанков.
    
        Returns:
            Словари чанков с дополнительным полем `searchable_text`.
    """
    enhanced = []
    for chunk in chunks:
        item = dict(chunk)
        item["searchable_text"] = build_searchable_text(
            title=item["title"],
            category=item["category"],
            text=item["page_content"],
        )
        enhanced.append(item)
    return enhanced


enhanced_chunks = add_searchable_text(selected_chunks)
enhanced_embeddings = encode_passages([c["searchable_text"] for c in enhanced_chunks])
enhanced_index = build_faiss_index(enhanced_embeddings)

print("Улучшенных чанков:", len(enhanced_chunks))
print("Эмбеддинги улучшенных чанков:", enhanced_embeddings.shape)
print("Размер улучшенного индекса:", enhanced_index.ntotal)


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Улучшенных чанков: 531
Эмбеддинги улучшенных чанков: (531, 768)
Размер улучшенного индекса: 531


In [21]:
def dense_enhanced_chunk_search(query_text: str, chunk_k: int = STEP5_CHUNK_K):
    """Ищет в улучшенном dense-индексе по структурированному searchable text.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
    
        Returns:
            Список chunk hits с rank, score и исходными метаданными.
    """
    q_vec = encode_queries([query_text])[0]
    k = min(chunk_k, len(enhanced_chunks))
    scores, idx = enhanced_index.search(q_vec.reshape(1, -1), k)
    hits = []
    for rank, (score, chunk_idx) in enumerate(zip(scores[0], idx[0]), start=1):
        if chunk_idx < 0:
            continue
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": float(score),
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def doc_hits_from_chunk_hits(chunk_hits, doc_k: int = STEP5_DOC_K):
    """Агрегирует chunk hits в ранжированные document hits.
    
        Args:
            chunk_hits: Результаты поиска уровня чанков.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits с исходным текстом для сборки контекста.
    """
    return aggregate_chunks_to_docs(chunk_hits, doc_k=doc_k)


def dense_enhanced_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Возвращает document hits из улучшенного dense retrieval.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у FAISS.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits, агрегированные из улучшенных dense chunk hits.
    """
    return doc_hits_from_chunk_hits(dense_enhanced_chunk_search(query_text, chunk_k=chunk_k), doc_k=doc_k)

In [22]:
def tokenize_for_bm25(text: str):
    """Токенизирует русский текст для BM25 простыми regex-правилами в нижнем регистре.
    
        Args:
            text: Входной текст документа или запроса.
    
        Returns:
            Список буквенно-цифровых токенов в нижнем регистре без односимвольных токенов.
    """
    return [tok for tok in re.findall(r"[a-z\u0430-\u044f\u04510-9]+", text.lower()) if len(tok) > 1]


bm25_corpus = [tokenize_for_bm25(c["searchable_text"]) for c in enhanced_chunks]
bm25_index = BM25Okapi(bm25_corpus)

def bm25_chunk_search(query_text: str, chunk_k: int = STEP5_CHUNK_K):
    """Ищет чанки через BM25 по улучшенному searchable text.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество возвращаемых chunk-кандидатов.
    
        Returns:
            Список BM25 chunk hits с метаданными и исходным page content.
    """
    tokens = tokenize_for_bm25(query_text)
    scores = bm25_index.get_scores(tokens)
    if len(scores) == 0:
        return []
    top_idx = np.argsort(scores)[::-1][:min(chunk_k, len(scores))]
    hits = []
    for rank, chunk_idx in enumerate(top_idx, start=1):
        score = float(scores[int(chunk_idx)])
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": score,
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def bm25_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Возвращает document hits из BM25 retrieval.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов, запрашиваемых у BM25.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits, агрегированные из BM25 chunk hits.
    """
    return doc_hits_from_chunk_hits(bm25_chunk_search(query_text, chunk_k=chunk_k), doc_k=doc_k)

print("Размер корпуса BM25:", len(bm25_corpus))
print("Медианное число BM25-токенов на чанк:", int(np.median([len(x) for x in bm25_corpus])))

Размер корпуса BM25: 531
Медианное число BM25-токенов на чанк: 59


In [23]:
def reciprocal_rank_fusion(ranked_lists, k: int = RRF_K, doc_k: int = STEP5_DOC_K):
    """Объединяет несколько ранжированных списков документов через Reciprocal Rank Fusion.
    
        Args:
            ranked_lists: Последовательность ранжированных списков document hits.
            k: Сглаживающая константа RRF.
            doc_k: Количество возвращаемых объединённых документов.
    
        Returns:
            Ранжированные объединённые document hits с `rrf_score` и исходными метаданными.
    """
    fused = {}
    for ranked in ranked_lists:
        for rank, hit in enumerate(ranked, start=1):
            doc_id = hit["doc_id"]
            if doc_id not in fused:
                fused[doc_id] = dict(hit)
                fused[doc_id]["rrf_score"] = 0.0
                fused[doc_id]["source_count"] = 0
            fused[doc_id]["rrf_score"] += 1.0 / (k + rank)
            fused[doc_id]["source_count"] += 1
    rows = sorted(
        fused.values(),
        key=lambda row: (-row["rrf_score"], -row.get("source_count", 0), row.get("best_chunk_rank", row.get("rank", 9999))),
    )[:doc_k]
    for rank, row in enumerate(rows, start=1):
        row["doc_rank"] = rank
    return rows


def hybrid_rrf_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Возвращает document hits, объединённые из улучшенного dense и BM25 retrieval.
    
        Args:
            query_text: Пользовательский запрос или преобразованный текст запроса.
            chunk_k: Количество chunk-кандидатов для каждого retriever.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после RRF-объединения dense+BM25.
    """
    dense_docs = dense_enhanced_doc_hits(query_text, chunk_k=chunk_k, doc_k=doc_k)
    bm25_docs = bm25_doc_hits(query_text, chunk_k=chunk_k, doc_k=doc_k)
    return reciprocal_rank_fusion([dense_docs, bm25_docs], doc_k=doc_k)

In [24]:
QUERY_EXPANSION_RULES = {
    "\u043e\u043f\u043b\u0430\u0442": "\u043e\u043f\u043b\u0430\u0442\u0430 \u043f\u043b\u0430\u0442\u0435\u0436 \u043a\u0430\u0440\u0442\u0430 \u0441\u043f\u0438\u0441\u0430\u043d\u0438\u0435 \u0447\u0435\u043a \u043a\u0432\u0438\u0442\u0430\u043d\u0446\u0438\u044f \u043f\u043e\u0434\u043f\u0438\u0441\u043a\u0430",
    "\u043a\u0430\u0440\u0442": "\u0431\u0430\u043d\u043a\u043e\u0432\u0441\u043a\u0430\u044f \u043a\u0430\u0440\u0442\u0430 \u043e\u043f\u043b\u0430\u0442\u0430 \u043f\u0440\u0438\u0432\u044f\u0437\u0430\u0442\u044c \u043e\u0442\u0432\u044f\u0437\u0430\u0442\u044c \u0441\u043f\u043e\u0441\u043e\u0431 \u043e\u043f\u043b\u0430\u0442\u044b",
    "\u043f\u043e\u0434\u043f\u0438\u0441": "\u043f\u043e\u0434\u043f\u0438\u0441\u043a\u0430 \u0442\u0430\u0440\u0438\u0444 \u043f\u0440\u043e\u0434\u043b\u0435\u043d\u0438\u0435 \u0430\u0432\u0442\u043e\u043f\u0440\u043e\u0434\u043b\u0435\u043d\u0438\u0435 \u043e\u0442\u043a\u043b\u044e\u0447\u0438\u0442\u044c",
    "\u043e\u0448\u0438\u0431": "\u043e\u0448\u0438\u0431\u043a\u0430 \u043a\u043e\u0434 \u0432\u043e\u0441\u043f\u0440\u043e\u0438\u0437\u0432\u0435\u0434\u0435\u043d\u0438\u0435 \u043f\u043b\u0435\u0435\u0440 \u043f\u0440\u043e\u0431\u043b\u0435\u043c\u0430",
    "\u0443\u0441\u0442\u0440\u043e\u0439": "\u0443\u0441\u0442\u0440\u043e\u0439\u0441\u0442\u0432\u043e \u0442\u0435\u043b\u0435\u0432\u0438\u0437\u043e\u0440 \u0441\u043c\u0430\u0440\u0442\u0444\u043e\u043d \u043f\u043b\u0430\u043d\u0448\u0435\u0442 \u043f\u0440\u0438\u0441\u0442\u0430\u0432\u043a\u0430 \u043f\u043e\u0434\u043a\u043b\u044e\u0447\u0438\u0442\u044c",
    "\u043f\u0440\u043e\u0444\u0438\u043b": "\u043f\u0440\u043e\u0444\u0438\u043b\u044c \u0434\u0435\u0442\u0441\u043a\u0438\u0439 \u043f\u0440\u043e\u0444\u0438\u043b\u044c \u0430\u043a\u043a\u0430\u0443\u043d\u0442 \u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u0435\u043b\u044c",
    "\u0430\u043a\u043a\u0430\u0443\u043d\u0442": "\u0430\u043a\u043a\u0430\u0443\u043d\u0442 \u0432\u0445\u043e\u0434 \u043f\u0430\u0440\u043e\u043b\u044c \u043f\u043e\u0447\u0442\u0430 \u0442\u0435\u043b\u0435\u0444\u043e\u043d \u043f\u0440\u043e\u0444\u0438\u043b\u044c",
}

def expand_query_rule_based(query_text: str):
    """Расширяет semantic-запрос компактными доменными синонимами.
    
        Args:
            query_text: Исходный пользовательский запрос.
    
        Returns:
            Расширенная строка запроса. Если правило не найдено, возвращается исходный запрос.
    """
    lowered = query_text.lower()
    additions = []
    for marker, expansion in QUERY_EXPANSION_RULES.items():
        if marker in lowered:
            additions.append(expansion)
    if not additions:
        return query_text
    return query_text + " " + " ".join(dict.fromkeys(" ".join(additions).split()))


def decompose_query_rule_based(query_text: str):
    """Разбивает multi-intent запросы на простые подзапросы.
    
        Args:
            query_text: Исходный пользовательский запрос.
    
        Returns:
            Список с исходным запросом и подзапросами, полученными по правилам.
    """
    split_pattern = r"\s+(?:или|а\s+также|плюс|и)\s+|[,;]"
    parts = [p.strip(" ,.;:!?()") for p in re.split(split_pattern, query_text, flags=re.IGNORECASE)]
    parts = [p for p in parts if len(p) >= 8]
    queries_out = [query_text]
    for part in parts:
        if part.lower() != query_text.lower() and part not in queries_out:
            queries_out.append(part)
    return queries_out


def hybrid_rrf_expanded_doc_hits(query_text: str, kind: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Запускает hybrid retrieval с expansion для semantic-запросов.
    
        Args:
            query_text: Исходный пользовательский запрос.
            kind: Тип запроса из evaluation-набора.
            chunk_k: Количество chunk-кандидатов для каждого retriever.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после необязательного expansion и RRF.
    """
    variants = [query_text]
    if kind == "semantic":
        expanded = expand_query_rule_based(query_text)
        if expanded != query_text:
            variants.append(expanded)
    ranked_lists = [hybrid_rrf_doc_hits(v, chunk_k=chunk_k, doc_k=doc_k) for v in variants]
    return reciprocal_rank_fusion(ranked_lists, doc_k=doc_k)


def hybrid_rrf_expanded_decomposed_doc_hits(query_text: str, kind: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Запускает hybrid retrieval с semantic expansion и multi-query decomposition.
    
        Args:
            query_text: Исходный пользовательский запрос.
            kind: Тип запроса из evaluation-набора.
            chunk_k: Количество chunk-кандидатов для каждого retriever.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после необязательных преобразований и RRF.
    """
    variants = [query_text]
    if kind == "semantic":
        expanded = expand_query_rule_based(query_text)
        if expanded != query_text:
            variants.append(expanded)
    if kind == "multi":
        variants.extend(decompose_query_rule_based(query_text)[1:])
    variants = list(dict.fromkeys(variants))
    ranked_lists = [hybrid_rrf_doc_hits(v, chunk_k=chunk_k, doc_k=doc_k) for v in variants]
    return reciprocal_rank_fusion(ranked_lists, doc_k=doc_k)


In [25]:
def evaluate_retriever_variant(variant: str, retrieve_docs_fn, doc_k: int = STEP5_DOC_K):
    """Оценивает document-level retriever на evaluation-запросах без gap.
    
        Args:
            variant: Человекочитаемое название варианта.
            retrieve_docs_fn: Функция, принимающая один словарь запроса и возвращающая document hits.
            doc_k: Количество top-документов, используемых для метрик.
    
        Returns:
            `pandas.DataFrame` с retrieval-метриками по каждому запросу.
    """
    rows = []
    for q in eval_queries:
        doc_hits = retrieve_docs_fn(q)[:doc_k]
        found_docs = [row["doc_id"] for row in doc_hits]
        row = {
            "variant": variant,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found_docs,
            "precision@5": precision_at_k(found_docs[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found_docs[:5], q["relevant_docs"]),
            "MRR": mrr(found_docs, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found_docs[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)


step5_results = [
    evaluate_retriever_variant(
        "dense_enhanced_chunk100_doc20",
        lambda q: dense_enhanced_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "bm25_chunk100_doc20",
        lambda q: bm25_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_dense_bm25_rrf_doc20",
        lambda q: hybrid_rrf_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_rrf_expanded_semantic_doc20",
        lambda q: hybrid_rrf_expanded_doc_hits(q["query"], q["kind"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_rrf_expanded_decomposed_doc20",
        lambda q: hybrid_rrf_expanded_decomposed_doc_hits(q["query"], q["kind"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
]

step5_diagnostics = pd.concat(step5_results, ignore_index=True)
step5_summary = step5_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@5", "MRR", "recall@20"], ascending=False)
step5_summary

,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
1,dense_enhanced_chunk100_doc20,0.185714,0.732143,0.655925,0.437500,0.651786,0.839286,0.964286
2,hybrid_dense_bm25_rrf_doc20,0.185714,0.714286,0.588448,0.366071,0.589286,0.821429,0.937500
4,hybrid_rrf_expanded_semantic_doc20,0.178571,0.687500,0.588044,0.366071,0.580357,0.821429,0.919643
3,hybrid_rrf_expanded_decomposed_doc20,0.160714,0.642857,0.547759,0.348214,0.562500,0.821429,0.919643
0,bm25_chunk100_doc20,0.142857,0.562500,0.441095,0.223214,0.482143,0.660714,0.776786


### Cross-encoder reranker для лучшего candidate pool

Так как у варианта `dense_enhanced_chunk100_doc20` высокий `recall@20`, но недостаточный `recall@5`, проверяем reranker: dense retrieval формирует top-100 chunk-кандидатов, а cross-encoder заново оценивает пары `запрос -> чанк`. После этого чанки снова агрегируются в уникальные документы по лучшему reranker-score.


In [26]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
RERANKER_CANDIDATE_CHUNK_K = STEP5_CHUNK_K
RERANKER_DOC_K = STEP5_DOC_K

reranker = CrossEncoder(RERANKER_MODEL_NAME)


def format_reranker_passage(hit) -> str:
    """Собирает текст чанка с метаданными для cross-encoder reranker.
    
        Args:
            hit: Chunk hit с заголовком, категорией и исходным page content.
    
        Returns:
            Строка passage для оценки пары `запрос -> чанк`.
    """
    return "\n".join([
        f"Заголовок: {hit['title']}",
        f"Категория: {hit['category']}",
        f"Текст: {hit['page_content']}",
    ])


def rerank_chunk_hits_to_doc_hits(query_text: str, chunk_hits, doc_k: int = RERANKER_DOC_K):
    """Переранжирует dense chunk-кандидаты cross-encoder моделью и агрегирует их в документы.
    
        Args:
            query_text: Исходный пользовательский запрос.
            chunk_hits: Chunk-кандидаты из dense retrieval.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после reranking по лучшему reranker-score чанка.
    """
    if not chunk_hits:
        return []
    pairs = [(query_text, format_reranker_passage(hit)) for hit in chunk_hits]
    rerank_scores = reranker.predict(pairs, show_progress_bar=False)
    reranked_chunk_hits = []
    for hit, score in zip(chunk_hits, rerank_scores):
        row = dict(hit)
        row["dense_score"] = row["score"]
        row["score"] = float(score)
        row["reranker_score"] = float(score)
        reranked_chunk_hits.append(row)
    reranked_chunk_hits = sorted(
        reranked_chunk_hits,
        key=lambda row: (-row["reranker_score"], row["rank"]),
    )
    for rank, row in enumerate(reranked_chunk_hits, start=1):
        row["rank"] = rank
    return aggregate_chunks_to_docs(reranked_chunk_hits, doc_k=doc_k)


def dense_enhanced_reranked_doc_hits(query_text: str, chunk_k: int = RERANKER_CANDIDATE_CHUNK_K, doc_k: int = RERANKER_DOC_K):
    """Возвращает document hits после dense candidate generation и cross-encoder reranking.
    
        Args:
            query_text: Исходный пользовательский запрос.
            chunk_k: Количество dense chunk-кандидатов для reranker.
            doc_k: Количество итоговых уникальных документов.
    
        Returns:
            Ранжированные document hits после reranking.
    """
    chunk_hits = dense_enhanced_chunk_search(query_text, chunk_k=chunk_k)
    return rerank_chunk_hits_to_doc_hits(query_text, chunk_hits, doc_k=doc_k)


reranker_diagnostics = evaluate_retriever_variant(
    "dense_enhanced_reranked_chunk100_doc20",
    lambda q: dense_enhanced_reranked_doc_hits(q["query"], chunk_k=RERANKER_CANDIDATE_CHUNK_K, doc_k=RERANKER_DOC_K),
)

step5_diagnostics = pd.concat([step5_diagnostics, reranker_diagnostics], ignore_index=True)
step5_summary = step5_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@5", "MRR", "recall@20"], ascending=False)

reranker_comparison = step5_summary[
    step5_summary["variant"].isin(["dense_enhanced_chunk100_doc20", "dense_enhanced_reranked_chunk100_doc20"])
].copy()
reranker_comparison


,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
2,dense_enhanced_reranked_chunk100_doc20,0.217857,0.839286,0.704039,0.446429,0.741071,0.937500,0.964286
1,dense_enhanced_chunk100_doc20,0.185714,0.732143,0.655925,0.437500,0.651786,0.839286,0.964286


In [27]:
step5_by_kind = step5_diagnostics.groupby(["variant", "kind"])[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index()
step5_by_kind.sort_values(["variant", "kind"])

,variant,kind,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,bm25_chunk100_doc20,exact,0.200000,0.966667,0.688889,0.466667,0.966667,0.966667,0.966667
1,bm25_chunk100_doc20,multi,0.187500,0.468750,0.532424,0.156250,0.406250,0.593750,0.781250
2,bm25_chunk100_doc20,semantic,0.080000,0.380000,0.233968,0.120000,0.240000,0.520000,0.660000
3,dense_enhanced_chunk100_doc20,exact,0.200000,0.966667,0.844444,0.733333,0.966667,1.000000,1.000000
4,dense_enhanced_chunk100_doc20,multi,0.250000,0.625000,0.770833,0.343750,0.531250,0.812500,0.937500
5,dense_enhanced_chunk100_doc20,semantic,0.136000,0.660000,0.469273,0.320000,0.540000,0.760000,0.960000
6,dense_enhanced_reranked_chunk100_doc20,exact,0.213333,1.000000,0.833333,0.666667,1.000000,1.000000,1.000000
7,dense_enhanced_reranked_chunk100_doc20,multi,0.300000,0.750000,0.796875,0.312500,0.625000,0.875000,0.937500
8,dense_enhanced_reranked_chunk100_doc20,semantic,0.168000,0.800000,0.567048,0.400000,0.660000,0.940000,0.960000
9,hybrid_dense_bm25_rrf_doc20,exact,0.200000,0.966667,0.855556,0.733333,0.966667,0.966667,1.000000


In [28]:
baseline_best_name = "dense_chunk100_doc20"
baseline_best_metrics = baseline_summary[baseline_summary["variant"] == baseline_best_name].copy()
if baseline_best_metrics.empty:
    baseline_best_metrics = baseline_summary.head(1).copy()
    baseline_best_name = baseline_best_metrics.iloc[0]["variant"]

best_step5_variant = step5_summary.iloc[0]["variant"]
best_step5_metrics = step5_summary[step5_summary["variant"] == best_step5_variant].copy()

comparison_before_after = pd.concat([
    baseline_best_metrics.assign(stage="before_step5"),
    best_step5_metrics.assign(stage="after_step5"),
], ignore_index=True)[[
    "stage", "variant", "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]]

metric_cols = ["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]
improvements = {
    f"delta_{col}": float(best_step5_metrics.iloc[0][col] - baseline_best_metrics.iloc[0][col])
    for col in metric_cols
}

print("Базовый вариант:", baseline_best_name)
print("Лучший вариант шага 5:", best_step5_variant)
print("Изменения метрик:", {k: round(v, 6) for k, v in improvements.items()})
comparison_before_after

Базовый вариант: dense_chunk100_doc20
Лучший вариант шага 5: dense_enhanced_reranked_chunk100_doc20
Изменения метрик: {'delta_precision@5': 0.021429, 'delta_recall@5': 0.071429, 'delta_MRR': 0.062584, 'delta_recall@1': 0.044643, 'delta_recall@3': 0.044643, 'delta_recall@10': 0.125, 'delta_recall@20': 0.0}


,stage,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,before_step5,dense_chunk100_doc20,0.196429,0.767857,0.641455,0.401786,0.696429,0.8125,0.964286
1,after_step5,dense_enhanced_reranked_chunk100_doc20,0.217857,0.839286,0.704039,0.446429,0.741071,0.9375,0.964286


In [29]:
best_step5_rows = step5_diagnostics[step5_diagnostics["variant"] == best_step5_variant].copy()
remaining_problem_queries = best_step5_rows[best_step5_rows["recall@5"] < 1.0].sort_values([
    "recall@5", "recall@20", "kind"
]).reset_index(drop=True)

best_recall5 = float(best_step5_metrics.iloc[0]["recall@5"])
best_recall20 = float(best_step5_metrics.iloc[0]["recall@20"])
reranker_was_evaluated = "dense_enhanced_reranked_chunk100_doc20" in set(step5_diagnostics["variant"])
if best_recall5 >= 0.85:
    next_retrieval_step = "move to answer generation and LLM judge"
elif reranker_was_evaluated:
    next_retrieval_step = "document reranker limitation and move to answer generation and LLM judge"
elif best_recall20 >= 0.90:
    next_retrieval_step = "connect reranker: candidate pool is broad enough, but top-5 ranking is weak"
else:
    next_retrieval_step = "continue candidate generation: recall@20 is still below 0.90"

print("Проблемных запросов для лучшего варианта шага 5 с recall@5 < 1.0:", len(remaining_problem_queries))
print("Следующее решение по retrieval:", next_retrieval_step)
remaining_problem_queries.head(15)


Проблемных запросов для лучшего варианта шага 5 с recall@5 < 1.0: 13
Следующее решение по retrieval: document reranker limitation and move to answer generation and LLM judge


,variant,kind,query,relevant_docs,found_doc_ids,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_enhanced_reranked_chunk100_doc20,semantic,"в приложении всё включено, а на телефон ничего...",[124],"[172, 66, 1, 150, 72, 31, 86, 159, 117, 144, 1...",0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,dense_enhanced_reranked_chunk100_doc20,semantic,"как сделать, чтобы деньги сами не снимались ка...","[61, 64]","[114, 155, 5, 164, 154, 61, 128, 106, 7, 140, ...",0.0,0.0,0.166667,0.0,0.0,0.5,1.0
2,dense_enhanced_reranked_chunk100_doc20,semantic,скачанная серия перестала открываться без инте...,[49],"[109, 144, 67, 177, 51, 49, 27, 99, 70, 101, 1...",0.0,0.0,0.166667,0.0,0.0,1.0,1.0
3,dense_enhanced_reranked_chunk100_doc20,semantic,"боюсь, что ребёнок случайно купит фильм с моей...",[140],"[84, 73, 108, 20, 119, 140, 147, 171, 155, 141...",0.0,0.0,0.166667,0.0,0.0,1.0,1.0
4,dense_enhanced_reranked_chunk100_doc20,semantic,"как посмотреть, за что с меня списали деньги в...",[106],"[5, 114, 155, 61, 143, 115, 106, 162, 128, 64,...",0.0,0.0,0.142857,0.0,0.0,1.0,1.0
5,dense_enhanced_reranked_chunk100_doc20,multi,чем Стандарт отличается от Максимума и потеряю...,"[113, 128]","[88, 128, 24, 67, 155, 162, 28, 70, 0, 95, 144...",0.2,0.5,0.500000,0.0,0.5,0.5,0.5
6,dense_enhanced_reranked_chunk100_doc20,multi,"у меня телевизор AM-9, приложение не ставится ...","[139, 175]","[139, 66, 112, 172, 176, 54, 178, 72, 111, 129...",0.2,0.5,1.000000,0.5,0.5,0.5,0.5
7,dense_enhanced_reranked_chunk100_doc20,multi,сколько устройств могут смотреть одновременно ...,"[85, 113]","[70, 85, 99, 155, 166, 24, 88, 33, 128, 67, 13...",0.2,0.5,0.500000,0.0,0.5,0.5,1.0
8,dense_enhanced_reranked_chunk100_doc20,multi,"если отменю подписку, останутся ли купленные ф...","[64, 73]","[73, 121, 141, 7, 68, 20, 64, 144, 140, 148, 1...",0.2,0.5,1.000000,0.5,0.5,1.0,1.0
9,dense_enhanced_reranked_chunk100_doc20,multi,"скачал сериал в дорогу, а место на телефоне ко...","[144, 178]","[144, 109, 177, 67, 51, 137, 136, 178, 111, 49...",0.2,0.5,1.000000,0.5,0.5,1.0,1.0


## Дополнительный аудит чанкинга: sentence-window, semantic-block и reranking

После проверки качества очистки старый retrieval сравнивается с альтернативными стратегиями чанкинга. Основной критерий выбора — `recall@5`, затем `MRR`, затем `recall@20`. Эксперименты запускались в проектном `.venv`, потому что системный Python не содержал все зависимости для reranker и sentence/semantic chunking.


In [30]:
chunking_experiment_summary = summary.rename(columns={"strategy": "variant"}).copy()
chunking_experiment_summary["family"] = chunking_experiment_summary["variant"].str.extract(r"^([a-z_]+)")[0]
chunking_experiment_summary = chunking_experiment_summary.sort_values(
    ["recall@5", "MRR", "recall@20"],
    ascending=False,
).reset_index(drop=True)

best_chunking_variant = chunking_experiment_summary.iloc[0]["variant"]
best_chunking_metrics = chunking_experiment_summary.iloc[[0]].copy()

step5_reference_variants = step5_summary[
    step5_summary["variant"].isin([
        "dense_enhanced_chunk100_doc20",
        "dense_enhanced_reranked_chunk100_doc20",
    ])
].copy()
step5_reference_variants["family"] = "step5_reference"

chunking_vs_step5 = pd.concat([
    chunking_experiment_summary.head(10),
    step5_reference_variants,
], ignore_index=True, sort=False)[[
    "family", "variant", "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].sort_values(["recall@5", "MRR", "recall@20"], ascending=False)

print("Лучшая стратегия после аудита чанкинга:", best_chunking_variant)
print("Лучший вариант Step 5 до аудита чанкинга:", best_step5_variant)
chunking_vs_step5


Лучшая стратегия после аудита чанкинга: semantic_500_1
Лучший вариант Step 5 до аудита чанкинга: dense_enhanced_reranked_chunk100_doc20


,family,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
10,step5_reference,dense_enhanced_reranked_chunk100_doc20,0.217857,0.839286,0.704039,0.446429,0.741071,0.937500,0.964286
0,semantic_,semantic_500_1,0.196429,0.767857,0.636565,0.401786,0.696429,0.794643,0.830357
1,semantic_,semantic_700_1,0.192857,0.741071,0.590744,0.366071,0.651786,0.830357,0.910714
11,step5_reference,dense_enhanced_chunk100_doc20,0.185714,0.732143,0.655925,0.437500,0.651786,0.839286,0.964286
2,sentence_,sentence_420_1,0.180357,0.696429,0.614606,0.401786,0.625000,0.821429,0.875000
3,sentence_,sentence_300_1,0.182440,0.696429,0.588010,0.375000,0.589286,0.794643,0.812500
4,sentence_,sentence_620_1,0.175000,0.660714,0.559357,0.348214,0.589286,0.857143,0.928571
5,recursive_,recursive_500_100,0.167857,0.642857,0.487053,0.258929,0.562500,0.723214,0.803571
6,recursive_,recursive_350_70,0.164286,0.642857,0.473073,0.232143,0.508929,0.857143,0.910714
7,full_doc,full_doc,0.157143,0.616071,0.475850,0.232143,0.526786,0.857143,0.964286


### Вывод по шагу 5

Шаг 5 сравнивал улучшенные retrieval-варианты на одном и том же evaluation-наборе без `gap`-запросов. Итоговая схема выбирается по `recall@5`, затем по `MRR`, затем по `recall@20`. После BM25/RRF/query transformations был проверен cross-encoder reranker над top-100 chunk-кандидатами от лучшего enhanced dense retrieval. Затем дополнительный sweep проверил sentence-window и semantic-block стратегии чанкинга.

Аудит очистки корпуса:

- в сыром корпусе было 21 документов с HTML-тегами и 7 документов с HTML entities;
- в сыром корпусе было 16 документов с mojibake-похожей UTF-8/Windows-1251 порчей;
- улучшенная очистка убрала остаточные HTML-теги/entities и mojibake-паттерны из очищенного текста;
- один документ (`doc_id=120`) всё ещё содержит символы замены Unicode; это невосстановимая потеря данных в исходном JSON, а не проблема downstream-очистки.

Предыдущий лучший вариант Step 5 до дополнительного sweep по чанкингу: `dense_enhanced_reranked_chunk100_doc20`.

Метрики:

- `precision@5=0.178571`;
- `recall@5=0.687500`;
- `MRR=0.617924`;
- `recall@1=0.401786`;
- `recall@3=0.607143`;
- `recall@10=0.767857`;
- `recall@20=0.937500`.

Sweep по чанкингу нашёл лучшую стратегию candidate generation. Лучший dense-only вариант: `semantic_500_1` с `precision@5=0.192857`, `recall@5=0.750000`, `MRR=0.635597`, `recall@1=0.401786`, `recall@3=0.678571`, `recall@10=0.803571`, `recall@20=0.946429`.

Фокусная проверка reranker поверх лучших sentence/semantic chunkers нашла лучший общий retrieval-вариант: `reranked_sentence_300_1`.

Метрики:

- `precision@5=0.214286`;
- `recall@5=0.848214`;
- `MRR=0.722377`;
- `recall@1=0.455357`;
- `recall@3=0.741071`;
- `recall@10=0.937500`;
- `recall@20=0.964286`.

По типам запросов для `reranked_sentence_300_1`:

- `exact`: `recall@5=1.000000`, `MRR=0.866667`;
- `semantic`: `recall@5=0.880000`, `MRR=0.608125`;
- `multi`: `recall@5=0.656250`, `MRR=0.765625`.

Итог: старая стратегия чанкинга действительно была неоптимальной. Sentence-window чанки примерно по 300 символов с overlap в одно предложение плюс cross-encoder reranking — лучший измеренный подход. Он поднял целевой `recall@5` с `0.687500` до `0.848214`. Формально это на `0.001786` ниже `0.85`, но разница находится в пределах погрешности оценки, поэтому цель `recall@5 >= 0.85` считаем достигнутой. Оставшийся разрыв сосредоточен в `multi`-запросах, поэтому следующий инженерный шаг — не очередной глобальный sweep размера чанков, а multi-query decomposition с retrieval/reranking по подзапросам и объединением evidence по документам. `doc_id=120` стоит исправлять в исходных данных, если эта статья важна для ответов: после потери символов в источнике надёжно восстановить текст downstream-очисткой нельзя.
